In [1]:
%matplotlib inline
import numpy as np
import pandas as pd

C:\Users\Tuins\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


First, we need to filter the dataset on what we need for our research. ONLY CHANGE THE INPUT VARIABLES!

In [2]:
#INPUT VARIABLES: Only change route_type, the rest will change automatically.

route_type = 2              # 2 for train, 3 for bus
if route_type == 2:
    Buffer = 3000                # in m
    speed = 300                  # in m/min
else:
    Buffer = 400                # in m
    speed = 70                  # in m/min

In [3]:
dataset = pd.read_csv('big_dataset_merged.csv')

In [4]:
day = 20261013                                       # random weekday
departure_time = "06:30:00" , "09:00:00"             # morning peak

dataset = dataset[dataset['route_type'] == route_type]
dataset = dataset[dataset['date'] == day]
dataset = dataset[(dataset["departure_time"] >= departure_time[0]) & (dataset["departure_time"] <= departure_time[1])]
#dataset.head()

In [5]:
small = dataset[['stop_name', 'station_id', 'stop_lat', 'stop_lon', 'trip_id', 'departure_time']]
small = small.drop_duplicates(subset=["station_id", "trip_id"])

#display(small.head())

In [6]:
stations = (small.groupby("station_id")
            .agg(stop_name=('stop_name', 'first'),
                 frequency=("trip_id", lambda s: s.nunique() / 2.5),
                 stop_lat=("stop_lat", "mean"),
                 stop_lon=("stop_lon", "mean"),)
            .reset_index())

stations["SWT"] = 0.5 * (60 / stations["frequency"])
stations["AWT"] = stations["SWT"] + 0.75

display(stations.head())
stations.shape

,station_id,stop_name,frequency,stop_lat,stop_lon,SWT,AWT
0,stoparea:17807,Grijpskerk,4.0,53.255805,6.310047,7.500000,8.250000
1,stoparea:17808,Groningen,31.6,53.210359,6.563926,0.949367,1.699367
2,stoparea:17818,Bedum,4.0,53.306839,6.592183,7.500000,8.250000
3,stoparea:17826,Groningen Noord,8.8,53.230032,6.556462,3.409091,4.159091
4,stoparea:17866,Winsum,4.8,53.330212,6.520280,6.250000,7.000000


(32, 7)

In [7]:
stations.to_csv('frequency.csv')

Now we need the distance matrix, we calculate which distances are below the buffer. Then we we compute the travel time.

In [8]:
distances = pd.read_csv('distance matrix selection.csv', index_col= 'ID')
distances = distances.where(distances <= Buffer, 0)
#distances = distances.dropna(how='all')

In [9]:
times = (distances / speed)
times = times.T
times.index.name = 'station_id'

In [10]:
#times.to_csv('times_per_centroid.csv')

Now, we merge the two tables and change the format from wide to long.

In [11]:
station_times = stations.merge(times, on='station_id')

station_times.head()

,station_id,stop_name,frequency,stop_lat,stop_lon,SWT,AWT,BK407,BK408,BK409,...,CH523,CH524,CH525,CH526,CH527,CI524,CI525,CI526,CJ525,CJ526
0,stoparea:17807,Grijpskerk,4.0,53.255805,6.310047,7.500000,8.250000,9.677027,9.625795,9.860135,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,stoparea:17808,Groningen,31.6,53.210359,6.563926,0.949367,1.699367,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,stoparea:17818,Bedum,4.0,53.306839,6.592183,7.500000,8.250000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,stoparea:17826,Groningen Noord,8.8,53.230032,6.556462,3.409091,4.159091,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,stoparea:17866,Winsum,4.8,53.330212,6.520280,6.250000,7.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
attr = ["station_id", "stop_name", "frequency", "stop_lat", "stop_lon", "SWT", "AWT"]
centroid_cols = [c for c in station_times.columns if c not in attr + ["id"]]

melted = station_times.melt(
    id_vars=attr,
    value_vars=centroid_cols,
    var_name="centroid_id",
    value_name="travel_times",)  

melted = melted.dropna(subset=["travel_times"])

melted = melted[["centroid_id"] + attr + ["travel_times"]].sort_values("centroid_id")
melted = melted.reset_index(drop=True)

melted.head()

,centroid_id,station_id,stop_name,frequency,stop_lat,stop_lon,SWT,AWT,travel_times
0,AA460,stoparea:18296,Uithuizermeeden,4.4,53.413969,6.720201,6.818182,7.568182,9.958421
1,AA461,stoparea:18296,Uithuizermeeden,4.4,53.413969,6.720201,6.818182,7.568182,9.598131
2,AA462,stoparea:18296,Uithuizermeeden,4.4,53.413969,6.720201,6.818182,7.568182,9.520124
3,AA463,stoparea:17870,Roodeschool,4.0,53.421030,6.758386,7.500000,8.250000,9.262365
4,AA463,stoparea:18296,Uithuizermeeden,4.4,53.413969,6.720201,6.818182,7.568182,9.731190


Now, we calculate TAT, EDF and AI and make a new table with only the centroid and AI.

In [13]:
melted['TAT'] = melted['travel_times'] + melted['AWT']
melted['EDF'] = 0.5 * ( 60 / melted['TAT'])
#melted.head(6)

In [14]:
cgroup = melted.groupby(by="centroid_id")["EDF"]

In [15]:
max_edf = cgroup.transform("max")
sum_edf = cgroup.transform("sum")

melted["AI"] = max_edf + 0.5 * (sum_edf - max_edf)

In [16]:
#melted.to_csv('centroids_full_table.csv')

In [17]:
result = melted[["centroid_id", "AI"]].drop_duplicates("centroid_id").reset_index(drop=True)
result.head(10)

,centroid_id,AI
0,AA460,1.711684
1,AA461,1.747609
2,AA462,1.755587
3,AA463,2.590705
4,AA464,1.822816
5,AA465,1.919029
6,AA466,1.984123
7,AA467,2.001193
8,AA468,1.964998
9,AA469,1.886232


In [18]:
#result.to_csv('centroids_AI.csv')